In [0]:
%pip install --upgrade numpy pandas pyarrow db-dtypes google-cloud-bigquery google-auth
dbutils.library.restartPython()

# 01 — Criação das Origens de Dados

Este notebook simula as fontes de dados do **Indicador Criança Alfabetizada** (INEP / Base dos Dados).

São criadas três origens com padrões distintos de ingestão:
1. **Dados estruturados**: UF, municípios e metas nacionais (padrão API)
2. **CDC (Change Data Capture)**: Atualizações de metas por UF ao longo do tempo
3. **Arquivos JSON**: Microdados do indicador por município (ingestão de arquivos)

> **Nota**: Em produção, os dados seriam obtidos diretamente da plataforma
> [Base dos Dados](https://basedosdados.org/) via biblioteca `basedosdados` + BigQuery.
> Esta simulação usa `spark.createDataFrame()` para compatibilidade com Databricks Serverless.

**Próximo passo**: executar `02_carga_camada_bronze.py`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
import json
import os

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IngesterDados") \
    .getOrCreate()


## 1. Origem Estruturada: Dimensão UF e Metas Nacionais

Simula resposta de API com as 27 Unidades Federativas e metas anuais do programa
**Compromisso Nacional Criança Alfabetizada** (meta: 100% até 2030).

In [0]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Workspace/Users/vinicius.tmiura@gmail.com/1AIST_projects/tech-challenge-02/secrets/aist-tech-challenge02-30d3d92fb7ed.json"

In [0]:
with open(os.environ["GOOGLE_APPLICATION_CREDENTIALS"]) as f:
    chave = json.load(f)

In [0]:
from google.cloud import bigquery
import os
from google.oauth2 import service_account

# Cria as credenciais
credentials = service_account.Credentials.from_service_account_info(chave)
# Aponta para o arquivo de credenciais
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 

client = bigquery.Client(credentials=credentials, project="aist-tech-challenge02")



In [0]:
query_uf = """
    SELECT *
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.uf`
    LIMIT 10
"""

query_municipio = """
    SELECT *
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.municipio`
    LIMIT 10
"""

query_dicionario = """SELECT *
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
"""

query_alunos = """SELECT * FROM `basedosdados.br_inep_avaliacao_alfabetizacao.alunos` LIMIT 10"""

query_meta_alf_mun = """SELECT * FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio` LIMIT 10"""

query_meta_alf_uf = """SELECT * FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf` LIMIT 10"""

query_meta_alf_br = """SELECT * FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil` LIMIT 10"""

# Retorna direto como DataFrame
df_uf = client.query(query_uf).to_dataframe()
df_municipio = client.query(query_municipio).to_dataframe()
df_alunos = client.query(query_alunos).to_dataframe()
df_alf_mun = client.query(query_meta_alf_mun).to_dataframe()
df_alf_uf = client.query(query_meta_alf_uf).to_dataframe()
df_alf_br = client.query(query_meta_alf_br).to_dataframe()
df_dict = client.query(query_dicionario).to_dataframe()

In [0]:
df_uf.head()

In [0]:
df_municipio.head()

In [0]:
df_alunos.head()

In [0]:
df_alf_mun.head()

In [0]:
df_alf_uf.head()

In [0]:
df_alf_br.head()

In [0]:
spark_df_alf_br = spark.createDataFrame(df_alf_br)

In [0]:
spark_df_alf_br.show()

In [0]:
spark_df_alf_br.write.saveAsTable("tc02.db_raw_data.alfabetizacao_br")

In [0]:
dados_uf = [
$0
    ("AC", "Acre",                 "Norte"),
    ("AL", "Alagoas",              "Nordeste"),
    ("AP", "Amapá",                "Norte"),
    ("AM", "Amazonas",             "Norte"),
    ("BA", "Bahia",                "Nordeste"),
    ("CE", "Ceará",                "Nordeste"),
    ("DF", "Distrito Federal",     "Centro-Oeste"),
    ("ES", "Espírito Santo",       "Sudeste"),
    ("GO", "Goiás",                "Centro-Oeste"),
    ("MA", "Maranhão",             "Nordeste"),
    ("MT", "Mato Grosso",          "Centro-Oeste"),
    ("MS", "Mato Grosso do Sul",   "Centro-Oeste"),
    ("MG", "Minas Gerais",         "Sudeste"),
    ("PA", "Pará",                 "Norte"),
    ("PB", "Paraíba",              "Nordeste"),
    ("PR", "Paraná",               "Sul"),
    ("PE", "Pernambuco",           "Nordeste"),
    ("PI", "Piauí",                "Nordeste"),
    ("RJ", "Rio de Janeiro",       "Sudeste"),
    ("RN", "Rio Grande do Norte",  "Nordeste"),
    ("RS", "Rio Grande do Sul",    "Sul"),
    ("RO", "Rondônia",             "Norte"),
    ("RR", "Roraima",              "Norte"),
    ("SC", "Santa Catarina",       "Sul"),
    ("SP", "São Paulo",            "Sudeste"),
    ("SE", "Sergipe",              "Nordeste"),
    ("TO", "Tocantins",            "Norte"),
]

schema_uf = "sigla_uf STRING, nome_uf STRING, regiao STRING"
df_uf = spark.createDataFrame(dados_uf, schema=schema_uf)

# Metas nacionais anuais do Compromisso Nacional Criança Alfabetizada
# Ponto de corte: 743 pontos na escala SAEB (Pesquisa Alfabetiza Brasil, 2023)
dados_meta_brasil = [
    (2019, 52.0, "Linha de base do programa"),
    (2020, 54.0, "Meta com impacto COVID-19"),
    (2021, 57.0, "Retomada pós-pandemia"),
    (2022, 61.0, "Meta de crescimento acelerado"),
    (2023, 65.0, "Meta revisada pelo MEC / Alfabetiza Brasil"),
]

schema_meta_brasil = "ano INT, meta_nacional DOUBLE, contexto STRING"
df_meta_brasil = spark.createDataFrame(dados_meta_brasil, schema=schema_meta_brasil)

display(df_uf)
display(df_meta_brasil)

## 2. Origem CDC: Metas por UF com Histórico de Revisões

Simula eventos de Change Data Capture de um banco transacional de metas estaduais.
Estados do Norte e Nordeste tiveram metas revisadas em 2022 após diagnóstico do MEC.

In [0]:
# Taxa base aproximada do Indicador Criança Alfabetizada por UF (2019, % alunos acima de 743pts SAEB)
metas_base_uf = {
    "AC": 48.0, "AL": 43.0, "AP": 44.0, "AM": 46.0, "BA": 52.0,
    "CE": 54.0, "DF": 68.0, "ES": 63.0, "GO": 60.0, "MA": 41.0,
    "MT": 57.0, "MS": 59.0, "MG": 62.0, "PA": 43.0, "PB": 50.0,
    "PR": 70.0, "PE": 53.0, "PI": 47.0, "RJ": 64.0, "RN": 54.0,
    "RS": 72.0, "RO": 50.0, "RR": 48.0, "SC": 76.0, "SP": 71.0,
    "SE": 50.0, "TO": 52.0,
}

ufs_revisao_2022 = {
    "AC", "AL", "AP", "AM", "BA", "CE", "MA", "PA", "PB", "PE", "PI", "RN", "RR", "RO", "SE", "TO"
}

eventos_cdc_meta_uf = []
for i, (sigla, meta_base) in enumerate(metas_base_uf.items()):
    eventos_cdc_meta_uf.append({
        "Op": "I",
        "tabela": "meta_alfabetizacao_uf",
        "commit_timestamp": f"2019-01-15T08:00:{i:02d}",
        "data": {"sigla_uf": sigla, "ano": 2019, "meta_uf": meta_base, "versao": 1},
    })
    if sigla in ufs_revisao_2022:
        eventos_cdc_meta_uf.append({
            "Op": "U",
            "tabela": "meta_alfabetizacao_uf",
            "commit_timestamp": f"2022-06-{(i % 28) + 1:02d}T10:00:00",
            "data": {"sigla_uf": sigla, "ano": 2019, "meta_uf": round(meta_base * 1.05, 1), "versao": 2},
        })

json_cdc_meta_uf = [json.dumps(ev) for ev in eventos_cdc_meta_uf]

schema_cdc_meta_uf = T.StructType([
    T.StructField("Op", T.StringType(), True),
    T.StructField("tabela", T.StringType(), True),
    T.StructField("commit_timestamp", T.StringType(), True),
    T.StructField("data", T.StructType([
        T.StructField("sigla_uf", T.StringType(), True),
        T.StructField("ano", T.IntegerType(), True),
        T.StructField("meta_uf", T.DoubleType(), True),
        T.StructField("versao", T.IntegerType(), True),
    ]), True),
])

df_json_cdc_meta_uf = spark.createDataFrame(
    [(linha,) for linha in json_cdc_meta_uf], "json_evento STRING"
)

df_raw_cdc_meta_uf = (
    df_json_cdc_meta_uf
    .select(F.from_json(F.col("json_evento"), schema_cdc_meta_uf).alias("ev"))
    .select("ev.*")
)

janela_cdc = Window.partitionBy("data.sigla_uf", "data.ano").orderBy(F.col("commit_timestamp").desc())

df_meta_uf_resolvida = (
    df_raw_cdc_meta_uf
    .filter(F.col("Op").isin("I", "U"))
    .withColumn("rank_cdc", F.row_number().over(janela_cdc))
    .filter(F.col("rank_cdc") == 1)
    .select(
        F.col("data.sigla_uf").alias("sigla_uf"),
        F.col("data.ano").cast("int").alias("ano"),
        F.col("data.meta_uf").alias("meta_uf"),
        F.col("data.versao").alias("versao_meta"),
        F.col("commit_timestamp").alias("ultima_atualizacao_meta"),
    )
)

print(f"Eventos CDC gerados: {len(json_cdc_meta_uf)}")
print(f"Metas UF após CDC (estado mais recente): {df_meta_uf_resolvida.count()}")
display(df_meta_uf_resolvida.orderBy("sigla_uf"))

## 3. Origem por Arquivo JSON: Indicador por Município

Simula ingestão de arquivos JSON exportados do INEP / Base dos Dados.
Contém os microdados do **Indicador Criança Alfabetizada** por município (2019–2023),
usando o ponto de corte de **743 pontos** na escala de proficiência do SAEB.

In [0]:
municipios_data = [
    # (id_municipio, nome_municipio, sigla_uf, populacao_estimada, capital)
    (1200401, "Rio Branco",          "AC", 413418,   True),
    (2704302, "Maceió",              "AL", 1025360,  True),
    (1600303, "Macapá",              "AP", 522353,   True),
    (1302603, "Manaus",              "AM", 2255903,  True),
    (1300300, "Itacoatiara",         "AM", 103293,   False),
    (2927408, "Salvador",            "BA", 2886698,  True),
    (2910727, "Feira de Santana",    "BA", 631817,   False),
    (2304400, "Fortaleza",           "CE", 2703391,  True),
    (2301000, "Juazeiro do Norte",   "CE", 279940,   False),
    (5300108, "Brasília",            "DF", 3094325,  True),
    (3205309, "Vitória",             "ES", 365855,   True),
    (3201308, "Cariacica",           "ES", 403983,   False),
    (5208707, "Goiânia",             "GO", 1555626,  True),
    (5201405, "Anápolis",            "GO", 395909,   False),
    (2111300, "São Luís",            "MA", 1115932,  True),
    (2103704, "Imperatriz",          "MA", 263504,   False),
    (5103403, "Cuiabá",              "MT", 660113,   True),
    (5002704, "Campo Grande",        "MS", 906092,   True),
    (3106200, "Belo Horizonte",      "MG", 2530701,  True),
    (3170206, "Uberlândia",          "MG", 706597,   False),
    (1501402, "Belém",               "PA", 1499641,  True),
    (1505536, "Santarém",            "PA", 305838,   False),
    (2507507, "João Pessoa",         "PB", 817511,   True),
    (4106902, "Curitiba",            "PR", 1963726,  True),
    (4115200, "Londrina",            "PR", 575377,   False),
    (2611606, "Recife",              "PE", 1653461,  True),
    (2607901, "Caruaru",             "PE", 375437,   False),
    (2211001, "Teresina",            "PI", 866300,   True),
    (3304557, "Rio de Janeiro",      "RJ", 6775561,  True),
    (3303500, "Niterói",             "RJ", 516981,   False),
    (2408102, "Natal",               "RN", 890480,   True),
    (4314902, "Porto Alegre",        "RS", 1488252,  True),
    (4305108, "Caxias do Sul",       "RS", 557418,   False),
    (1100205, "Porto Velho",         "RO", 545597,   True),
    (1400100, "Boa Vista",           "RR", 437010,   True),
    (4209102, "Florianópolis",       "SC", 537211,   True),
    (4202404, "Blumenau",            "SC", 372123,   False),
    (3550308, "São Paulo",           "SP", 12325232, True),
    (3509502, "Campinas",            "SP", 1213792,  False),
    (3548708, "Ribeirão Preto",      "SP", 718069,   False),
    (2800308, "Aracaju",             "SE", 672613,   True),
    (1721000, "Palmas",              "TO", 319884,   True),
]

# Taxa base de alfabetização por UF (aprox. 2019 — com base no desempenho SAEB histórico)
taxa_base_uf = {
    "AC": 0.48, "AL": 0.43, "AP": 0.44, "AM": 0.46, "BA": 0.52,
    "CE": 0.54, "DF": 0.70, "ES": 0.63, "GO": 0.61, "MA": 0.41,
    "MT": 0.57, "MS": 0.59, "MG": 0.62, "PA": 0.43, "PB": 0.50,
    "PR": 0.70, "PE": 0.53, "PI": 0.47, "RJ": 0.64, "RN": 0.54,
    "RS": 0.73, "RO": 0.50, "RR": 0.49, "SC": 0.76, "SP": 0.71,
    "SE": 0.50, "TO": 0.52,
}

PONTO_CORTE_SAEB = 743
ANOS = [2019, 2020, 2021, 2022, 2023]

indicadores_municipio = []
for id_mun, nome_mun, sigla, pop, capital in municipios_data:
    taxa_uf = taxa_base_uf.get(sigla, 0.55)
    bonus_capital = 0.03 if capital else 0.0
    variacao_municipio = (id_mun % 100) / 1000.0

    for idx_ano, ano in enumerate(ANOS):
        melhoria_anual = idx_ano * 0.018
        taxa = min(0.97, taxa_uf + bonus_capital + melhoria_anual + variacao_municipio)
        total_alunos = max(50, int(pop * 0.012) + (id_mun % 300))
        alunos_alfa = int(total_alunos * taxa)
        indicador = round((alunos_alfa / total_alunos) * 100, 2)

        indicadores_municipio.append({
            "id_municipio": id_mun,
            "nome_municipio": nome_mun,
            "sigla_uf": sigla,
            "ano": ano,
            "total_alunos_2o_ano": total_alunos,
            "alunos_alfabetizados": alunos_alfa,
            "indicador_crianca_alfabetizada": indicador,
            "ponto_corte_saeb": PONTO_CORTE_SAEB,
            "fonte": "INEP_BASE_DADOS_SIMULADO",
        })

# Introduz duplicata intencional para demonstrar deduplicação na Silver
indicadores_municipio.append({
    "id_municipio": 3550308,
    "nome_municipio": "São Paulo",
    "sigla_uf": "SP",
    "ano": 2023,
    "total_alunos_2o_ano": 185234,
    "alunos_alfabetizados": 145000,
    "indicador_crianca_alfabetizada": 78.28,
    "ponto_corte_saeb": PONTO_CORTE_SAEB,
    "fonte": "INEP_BASE_DADOS_SIMULADO_DUPLICATA",
})

# Introduz registro com valor ausente para demonstrar tratamento na Silver
indicadores_municipio.append({
    "id_municipio": 9999999,
    "nome_municipio": None,
    "sigla_uf": "XX",
    "ano": 2023,
    "total_alunos_2o_ano": None,
    "alunos_alfabetizados": None,
    "indicador_crianca_alfabetizada": None,
    "ponto_corte_saeb": PONTO_CORTE_SAEB,
    "fonte": "INEP_BASE_DADOS_SIMULADO_INVALIDO",
})

json_indicadores = [json.dumps(r) for r in indicadores_municipio]

schema_indicador = T.StructType([
    T.StructField("id_municipio", T.IntegerType(), True),
    T.StructField("nome_municipio", T.StringType(), True),
    T.StructField("sigla_uf", T.StringType(), True),
    T.StructField("ano", T.IntegerType(), True),
    T.StructField("total_alunos_2o_ano", T.IntegerType(), True),
    T.StructField("alunos_alfabetizados", T.IntegerType(), True),
    T.StructField("indicador_crianca_alfabetizada", T.DoubleType(), True),
    T.StructField("ponto_corte_saeb", T.IntegerType(), True),
    T.StructField("fonte", T.StringType(), True),
])

df_json_indicadores = spark.createDataFrame(
    [(linha,) for linha in json_indicadores], "json_linha STRING"
)

df_municipio = spark.createDataFrame(
    municipios_data,
    schema="id_municipio INT, nome_municipio STRING, sigla_uf STRING, populacao_estimada INT, capital BOOLEAN",
)

print(f"Total de registros gerados (com duplicata e inválido): {len(indicadores_municipio)}")
display(df_json_indicadores.limit(5))
display(df_municipio.limit(5))

## 4. Persistência das Origens no Schema `origens`

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS origens")

tabelas_origem = {
    "origens.tc02_uf": df_uf
        .withColumn("_sistema_origem", F.lit("base_dados_api_simulado"))
        .withColumn("_data_criacao_origem", F.current_timestamp()),

    "origens.tc02_meta_brasil": df_meta_brasil
        .withColumn("_sistema_origem", F.lit("inep_meta_nacional_simulado"))
        .withColumn("_data_criacao_origem", F.current_timestamp()),

    "origens.tc02_cdc_meta_uf_eventos_json": df_json_cdc_meta_uf
        .withColumn("_sistema_origem", F.lit("cdc_dms_meta_uf_simulado"))
        .withColumn("_data_criacao_origem", F.current_timestamp()),

    "origens.tc02_indicador_municipio_json": df_json_indicadores
        .withColumn("_sistema_origem", F.lit("arquivo_json_inep_simulado"))
        .withColumn("_data_criacao_origem", F.current_timestamp()),

    "origens.tc02_municipio": df_municipio
        .withColumn("_sistema_origem", F.lit("ibge_municipios_simulado"))
        .withColumn("_data_criacao_origem", F.current_timestamp()),
}

for nome_tabela, df in tabelas_origem.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )

print("Tabelas de origem criadas com sucesso:")
for nome_tabela in tabelas_origem:
    print(f"  - {nome_tabela} => {spark.read.table(nome_tabela).count()} linhas")

print("\nPróximo passo: executar 02_carga_camada_bronze.py")